In [51]:
import pandas as pd
from rdkit import Chem
import torch
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
from sklearn.metrics import roc_auc_score
from torch_geometric.loader import DataLoader
import numpy as np
import random
from itertools import product
from sklearn.metrics import roc_auc_score, balanced_accuracy_score
from sklearn.model_selection import train_test_split

import sys
import os
sys.path.append(os.path.abspath('../'))
from reduceGraph import mol_to_graph, graph_to_pyg_oh, reduce_graph_from_mol_oh, mol_to_pool_idx
from networks import GAT, PPGAT

In [2]:
#DATASET 

dataset = pd.read_csv('clintox.csv')
dataset

,smiles,FDA_APPROVED,CT_TOX
0,*C(=O)[C@H](CCCCNC(=O)OCCOC)NC(=O)OCCOC,1,0
1,[C@@H]1([C@@H]([C@@H]([C@H]([C@@H]([C@@H]1Cl)C...,1,0
2,[C@H]([C@@H]([C@@H](C(=O)[O-])O)O)([C@H](C(=O)...,1,0
3,[H]/[NH+]=C(/C1=CC(=O)/C(=C\C=c2ccc(=C([NH3+])...,1,0
4,[H]/[NH+]=C(\N)/c1ccc(cc1)OCCCCCOc2ccc(cc2)/C(...,1,0
...,...,...,...
1479,O[Si](=O)O,1,0
1480,O=[Ti]=O,1,0
1481,O=[Zn],1,0
1482,OCl(=O)(=O)=O,1,0


In [3]:
dataset.FDA_APPROVED.value_counts()

FDA_APPROVED
1    1390
0      94
Name: count, dtype: int64

In [4]:
dataset.CT_TOX.value_counts()

CT_TOX
0    1372
1     112
Name: count, dtype: int64

In [6]:
tox = dataset['CT_TOX']
tox.value_counts()

CT_TOX
0    1372
1     112
Name: count, dtype: int64

In [45]:
def smiles_to_data(smiles, tox_label):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None  
    graph = mol_to_graph(mol)   # mol to networkx 
    data = graph_to_pyg_oh(graph)  # networx graph to pytorch geometric graph
    tox_label_tensor = torch.tensor([tox_label], dtype=torch.float) 
    data.y = torch.stack([tox_label_tensor], dim=1)

    return data


def dataframe_to_pyg_dataset(df, smiles_col, tox_label_col):
    data_list = []
    
    for idx, row in df.iterrows():
        smiles = row[smiles_col]
        tox_label = row[tox_label_col]
        data = smiles_to_data(smiles, tox_label)
        if data is not None:
            data.smiles = smiles
            data_list.append(data)

    return data_list


In [46]:
clintox_dataset = dataframe_to_pyg_dataset(dataset, 'smiles', 'CT_TOX')


[13:41:15] Explicit valence for atom # 0 N, 5, is greater than permitted
[13:41:15] Can't kekulize mol.  Unkekulized atoms: 9
[13:41:17] Explicit valence for atom # 10 N, 4, is greater than permitted
[13:41:17] Explicit valence for atom # 10 N, 4, is greater than permitted
[13:41:17] Can't kekulize mol.  Unkekulized atoms: 4
[13:41:17] Can't kekulize mol.  Unkekulized atoms: 4


In [47]:
#grid search
learning_rates = [1e-4, 5e-4, 1e-3]
batch_sizes = [16, 32, 64]
grid = list(product(learning_rates, batch_sizes))

In [60]:


def train_eval_model(mod, dataset, in_channels, edge_attr_dim, out_channels, grid, epochs=30):
    labels = [int(data.y.item()) for data in dataset]

    train_val_indices, test_indices = train_test_split(
        list(range(len(dataset))),
        test_size=0.2,
        stratify=labels,
        random_state=42
    )

    train_val_labels = [labels[i] for i in train_val_indices]

    # 10% of full = 12.5% of 80% → 0.125
    train_indices, val_indices = train_test_split(
        train_val_indices,
        test_size=0.125,
        stratify=train_val_labels,
        random_state=42
    )

    # Create subsets 
    train = [dataset[i] for i in train_indices]
    val = [dataset[i] for i in val_indices]
    test  = [dataset[i] for i in test_indices]


    #  Hyperparameter tuning 
    best_val_loss = float('inf')
    best_config = None
    best_model_state = None

    for lr, batch_size in grid:
        model = mod(in_channels, edge_attr_dim, hidden_channels=64, out_channels=out_channels).to('cuda')
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        criterion = torch.nn.BCEWithLogitsLoss()

        train_loader = DataLoader(train, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val, batch_size=batch_size)

        for epoch in range(epochs):
            model.train()
            for batch in train_loader:
                batch = batch.to('cuda')
                optimizer.zero_grad()
                out = model(batch)
                loss = criterion(out, batch.y.float())
                loss.backward()
                optimizer.step()

        # Evaluate on val
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                batch = batch.to('cuda')
                out = model(batch)
                val_loss += criterion(out, batch.y.float()).item() * batch.num_graphs


        val_loss /= len(val)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_config = (lr, batch_size)
            best_model_state = model.state_dict()

    #  Retrain final model on train + val 
    final_model = mod(in_channels, edge_attr_dim, hidden_channels=64, out_channels=out_channels).to('cuda')
    final_model.load_state_dict(best_model_state)

    full_train = train + val
    train_loader = DataLoader(full_train, batch_size=best_config[1], shuffle=True)
    optimizer = torch.optim.Adam(final_model.parameters(), lr=best_config[0])
    criterion = torch.nn.BCEWithLogitsLoss()

    for epoch in range(epochs):
        final_model.train()
        for batch in train_loader:
            batch = batch.to('cuda')
            optimizer.zero_grad()
            out = final_model(batch)
            loss = criterion(out, batch.y.float())
            loss.backward()
            optimizer.step()

    return final_model, best_config, best_val_loss

In [61]:
in_channels = clintox_dataset[0].x.size(-1)
edge_attr_dim = clintox_dataset[0].edge_attr.size(-1)
out_channels = 1

model, config, loss = train_eval_model(GAT, clintox_dataset, in_channels, edge_attr_dim, out_channels, grid)

print(config )





(0.0005, 32)


In [64]:
def cross_validate_stratified(
    mod,
    dataset, in_channels, edge_attr_dim, out_channels,
    best_lr, best_batch_size, epochs=30, k=5
):
    from sklearn.model_selection import StratifiedKFold
    from sklearn.metrics import roc_auc_score, balanced_accuracy_score
    import numpy as np
    import torch

    # Extract single target per sample
    y_vector = torch.stack([data.y.view(-1)[0] for data in dataset]).numpy()

    splitter = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)

    all_aurocs = []
    all_bal_accs = []

    for fold, (train_val_idx, test_idx) in enumerate(splitter.split(np.zeros(len(y_vector)), y_vector)):
        print(f"\n Fold {fold + 1}")

        # Split data
        y_train_val = y_vector[train_val_idx]
        train_val_data = [dataset[i] for i in train_val_idx]
        test_data = [dataset[i] for i in test_idx]

        # Inner val split
        inner_split = StratifiedKFold(n_splits=5, shuffle=True, random_state=fold)
        train_idx, val_idx = next(inner_split.split(np.zeros(len(y_train_val)), y_train_val))
        train = [train_val_data[i] for i in train_idx]
        val = [train_val_data[i] for i in val_idx]

        # Initialize model
        model = mod(in_channels, edge_attr_dim, hidden_channels=64, out_channels=out_channels).to('cuda')
        optimizer = torch.optim.Adam(model.parameters(), lr=best_lr)
        criterion = torch.nn.BCEWithLogitsLoss()

        train_loader = DataLoader(train, batch_size=best_batch_size, shuffle=True)
        val_loader = DataLoader(val, batch_size=best_batch_size)
        test_loader = DataLoader(test_data, batch_size=best_batch_size)

        # Training loop
        for epoch in range(epochs):
            model.train()
            for batch in train_loader:
                batch = batch.to('cuda')
                optimizer.zero_grad()
                out = model(batch)

                y_true = batch.y.float()
                if y_true.dim() == 1:
                    y_true = y_true.unsqueeze(1)

                loss = criterion(out, y_true)
                loss.backward()
                optimizer.step()

        # Evaluate on test set
        model.eval()
        y_true, y_scores = [], []

        with torch.no_grad():
            for batch in test_loader:
                batch = batch.to('cuda')
                out = model(batch)

                y_scores.append(out.cpu())
                y_true.append(batch.y.cpu())

        y_scores = torch.cat(y_scores).numpy()
        y_true = torch.cat(y_true).numpy()

        # AUROC
        auroc = roc_auc_score(y_true, y_scores)
        all_aurocs.append(auroc)

        # Balanced accuracy
        y_probs = 1 / (1 + np.exp(-y_scores))   # sigmoid
        y_pred = (y_probs >= 0.5).astype(int)

        bal_acc = balanced_accuracy_score(y_true, y_pred)
        all_bal_accs.append(bal_acc)

        print(f" Fold {fold + 1} AUROC: {auroc:.4f}")
        print(f" Fold {fold + 1} Balanced Acc: {bal_acc:.4f}")

    mean_auroc = np.mean(all_aurocs)
    std_auroc = np.std(all_aurocs)

    mean_bal_acc = np.mean(all_bal_accs)
    std_bal_acc = np.std(all_bal_accs)

    print(f"\n Mean AUROC = {mean_auroc:.4f} ± {std_auroc:.4f}")
    print(f" Mean Balanced Accuracy = {mean_bal_acc:.4f} ± {std_bal_acc:.4f}")

    return (
        all_aurocs, mean_auroc, std_auroc,
        all_bal_accs, mean_bal_acc, std_bal_acc
    )

In [67]:
all_aurocs, mean_aurocs, std_aurocs, all_bal_accs, mean_bal_accs, std_bal_accs = cross_validate_stratified(
    GAT,
    clintox_dataset,
    in_channels,
    edge_attr_dim,
    out_channels=1,
    best_lr=config[0],
    best_batch_size=config[1],
    epochs=100,
    k=5
)


 Fold 1
 Fold 1 AUROC: 0.8398
 Fold 1 Balanced Acc: 0.6262

 Fold 2


/tmp/ipykernel_1001315/2383589628.py:78: RuntimeWarning: overflow encountered in exp
  y_probs = 1 / (1 + np.exp(-y_scores))   # sigmoid


 Fold 2 AUROC: 0.8954
 Fold 2 Balanced Acc: 0.7371

 Fold 3


/tmp/ipykernel_1001315/2383589628.py:78: RuntimeWarning: overflow encountered in exp
  y_probs = 1 / (1 + np.exp(-y_scores))   # sigmoid


 Fold 3 AUROC: 0.8035
 Fold 3 Balanced Acc: 0.6137

 Fold 4


/tmp/ipykernel_1001315/2383589628.py:78: RuntimeWarning: overflow encountered in exp
  y_probs = 1 / (1 + np.exp(-y_scores))   # sigmoid


 Fold 4 AUROC: 0.8578
 Fold 4 Balanced Acc: 0.6862

 Fold 5
 Fold 5 AUROC: 0.8760
 Fold 5 Balanced Acc: 0.7379

 Mean AUROC = 0.8545 ± 0.0315
 Mean Balanced Accuracy = 0.6802 ± 0.0528


# RG

In [70]:
def smiles_to_rgdata(smiles, tox_label):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None  
    
    data = reduce_graph_from_mol_oh(mol)  # Convert to PyG reduced graph
    
    # Skip if edge_attr is missing 
    if data.edge_attr is None or data.edge_attr.dim() != 2 or data.edge_attr.size(0) == 0 or data.edge_attr.size(1) == 0:
        return None

    tox_label_tensor = torch.tensor([tox_label], dtype=torch.float) 
    data.y = torch.stack([tox_label_tensor], dim=1)
    return data

def dataframe_to_rg_pyg_dataset(df, smiles_col, tox_label_col):
    data_list = []

    for idx, row in df.iterrows():
        smiles = row[smiles_col]
        tox_label = row[tox_label_col]
        data = smiles_to_rgdata(smiles, tox_label)
        if data is not None:
            data.smiles = smiles
            data_list.append(data)

    return data_list

In [72]:
clintox_rg_dataset = dataframe_to_rg_pyg_dataset(dataset, 'smiles', 'CT_TOX')


[14:56:26] Explicit valence for atom # 0 N, 5, is greater than permitted
[14:57:36] Can't kekulize mol.  Unkekulized atoms: 9
[15:00:26] Explicit valence for atom # 10 N, 4, is greater than permitted
[15:00:26] Explicit valence for atom # 10 N, 4, is greater than permitted
[15:01:26] Can't kekulize mol.  Unkekulized atoms: 4
[15:01:26] Can't kekulize mol.  Unkekulized atoms: 4


In [74]:

in_channels = clintox_dataset[0].x.size(-1)
edge_attr_dim = clintox_dataset[0].edge_attr.size(-1)
out_channels = 1

model, config, loss = train_eval_model(GAT, clintox_rg_dataset, in_channels, edge_attr_dim, out_channels, grid)

print(config )


(0.0005, 16)


In [75]:
all_aurocs, mean_aurocs, std_aurocs, all_bal_accs, mean_bal_accs, std_bal_accs = cross_validate_stratified(
    GAT,
    clintox_rg_dataset,
    in_channels,
    edge_attr_dim,
    out_channels=1,
    best_lr=config[0],
    best_batch_size=config[1],
    epochs=100,
    k=5
)


 Fold 1


/tmp/ipykernel_1001315/2383589628.py:78: RuntimeWarning: overflow encountered in exp
  y_probs = 1 / (1 + np.exp(-y_scores))   # sigmoid


 Fold 1 AUROC: 0.7716
 Fold 1 Balanced Acc: 0.5971

 Fold 2
 Fold 2 AUROC: 0.7727
 Fold 2 Balanced Acc: 0.5861

 Fold 3
 Fold 3 AUROC: 0.7870
 Fold 3 Balanced Acc: 0.5480

 Fold 4
 Fold 4 AUROC: 0.8039
 Fold 4 Balanced Acc: 0.6717

 Fold 5
 Fold 5 AUROC: 0.6959
 Fold 5 Balanced Acc: 0.5098

 Mean AUROC = 0.7662 ± 0.0371
 Mean Balanced Accuracy = 0.5825 ± 0.0541


# PPGAT

In [76]:
def smiles_to_rgnn_data(smiles, tox_label):
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None  
    G = mol_to_graph(mol)   # mol to networkx 
    data = graph_to_pyg_oh(G)  # networx graph to pytorch geometric graph
    tox_label_tensor = torch.tensor([tox_label], dtype=torch.float) 
    data.y = torch.stack([tox_label_tensor], dim=1)
    
    pharma_index, new_edge_index, new_edge_attr = mol_to_pool_idx(mol)
    data.pharma_index = pharma_index
    data.new_edge_index = new_edge_index
    data.new_edge_attr = new_edge_attr

    # Skip if edge_attr is missing 
    if data.edge_attr is None or data.edge_attr.dim() != 2 or data.edge_attr.size(0) == 0 or data.edge_attr.size(1) == 0:
        return None
    
    if data.edge_index is None or data.edge_index.dim() != 2 or data.edge_index.size(0) == 0 or data.edge_index.size(1) == 0:
        return None

 

    return data

def dataframe_to_rgnn_pyg_dataset(df, smiles_col, tox_label_col):
    data_list = []

    for idx, row in df.iterrows():
        smiles = row[smiles_col]
    
        tox_label = row[tox_label_col]
        data = smiles_to_rgnn_data(smiles, tox_label)
        if data is not None:
            data.smiles = smiles
            data_list.append(data)

    return data_list

In [77]:
clintox_ppgat_dataset = dataframe_to_rgnn_pyg_dataset(dataset, 'smiles',  'CT_TOX')


[15:13:28] Explicit valence for atom # 0 N, 5, is greater than permitted
[15:13:29] Can't kekulize mol.  Unkekulized atoms: 9
[15:13:33] Explicit valence for atom # 10 N, 4, is greater than permitted
[15:13:33] Explicit valence for atom # 10 N, 4, is greater than permitted
[15:13:35] Can't kekulize mol.  Unkekulized atoms: 4
[15:13:35] Can't kekulize mol.  Unkekulized atoms: 4


In [79]:

in_channels = clintox_ppgat_dataset[0].x.size(-1)
edge_attr_dim = clintox_ppgat_dataset[0].edge_attr.size(-1)
out_channels = 1

model, config, loss = train_eval_model(PPGAT, clintox_ppgat_dataset, in_channels, edge_attr_dim, out_channels, grid)

print(config )


(0.001, 64)


In [80]:
all_aurocs, mean_aurocs, std_aurocs, all_bal_accs, mean_bal_accs, std_bal_accs = cross_validate_stratified(
    PPGAT,
    clintox_ppgat_dataset,
    in_channels,
    edge_attr_dim,
    out_channels=1,
    best_lr=config[0],
    best_batch_size=config[1],
    epochs=100,
    k=5
)


 Fold 1
 Fold 1 AUROC: 0.8714
 Fold 1 Balanced Acc: 0.7318

 Fold 2
 Fold 2 AUROC: 0.8263
 Fold 2 Balanced Acc: 0.6448

 Fold 3
 Fold 3 AUROC: 0.8388
 Fold 3 Balanced Acc: 0.6408

 Fold 4
 Fold 4 AUROC: 0.8537
 Fold 4 Balanced Acc: 0.7225

 Fold 5
 Fold 5 AUROC: 0.8452
 Fold 5 Balanced Acc: 0.6507

 Mean AUROC = 0.8471 ± 0.0151
 Mean Balanced Accuracy = 0.6781 ± 0.0403
